In [1]:
!pip -q install pyspark

In [2]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
from pathlib import Path
spark = SparkSession.builder.appName("StreamingSet4").getOrCreate()
uploaded = files.upload()


Saving content_catalog_120.csv to content_catalog_120.csv
Saving device_master_8.csv to device_master_8.csv
Saving subscriber_master_2400.csv to subscriber_master_2400.csv
Saving subscription_plans_4.csv to subscription_plans_4.csv
Saving watch_sessions_80000.csv to watch_sessions_80000.csv
Saving watch_sessions_extra_10000.csv to watch_sessions_extra_10000.csv


## Load datasets

In [3]:
def read_csv(name):
    return spark.read.csv(name, header=True, inferSchema=True)
sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")


In [4]:
for name, df in [("sessions_df", sessions_df), ("extra_df", extra_df), ("subscribers_df", subscribers_df), ("content_df", content_df), ("devices_df", devices_df), ("plans_df", plans_df)]:
    print(name, df.count())
    df.printSchema()
sessions_df.show(5, truncate=False)


sessions_df 80000
root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

extra_df 10000
root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_

## Part A — DataFrame Operations

### 01

In [5]:
sessions_df.printSchema()
print("Rows:", sessions_df.count())
sessions_df.show(10, truncate=False)
for field in sessions_df.schema.fields:
    print(field.name, "->", field.dataType.simpleString())

root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

Rows: 80000
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+--------

### 02

In [6]:
exercise_02_df = sessions_df.select("session_id", "subscriber_id", "content_id", "watch_minutes", "session_status").filter((F.col("session_status") == "Completed") & (F.col("watch_minutes") > 60))
exercise_02_df.show(20, truncate=False)

+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|SUB01859     |CT0075    |125          |Completed     |
|WS00000179|SUB00935     |CT0054    |62           |Completed     |
|WS00000184|SUB00072     |CT0119    |91           |Completed     |
|WS00000222|SUB01603     |CT0103    |116          |Completed     |
|WS00000233|SUB01233     |CT0027    |158          |Completed     |
|WS00000244|SUB00985     |CT0048    |140          |Completed     |
|WS00000249|SUB00681     |CT0100    |120          |Completed     |
|WS00000257|SUB02115     |CT0120    |68           |Completed     |
|WS00000292|SUB01068     |CT0048    |141          |Completed     |
|WS00000326|SUB00368     |CT0096    |97           |Completed     |
|WS00000415|SUB01758     |CT0105    |150          |Completed     |
|WS00000419|SUB00367     |CT0036    |99           |Completed  

### 03 — Create watch_hours and high_engagement

In [7]:
exercise_03_df = sessions_df.withColumn("watch_hours", F.col("watch_minutes") / 60).withColumn("high_engagement", F.when(F.col("completion_pct").isNull(), F.lit(None).cast("boolean")).otherwise(F.col("completion_pct") >= 75))
exercise_03_df.select("session_id", "watch_minutes", "watch_hours", "completion_pct", "high_engagement").show(20, truncate=False)

+----------+-------------+-------------------+--------------+---------------+
|session_id|watch_minutes|watch_hours        |completion_pct|high_engagement|
+----------+-------------+-------------------+--------------+---------------+
|WS00000001|6            |0.1                |11.8          |false          |
|WS00000002|31           |0.5166666666666667 |56.4          |false          |
|WS00000003|29           |0.48333333333333334|63.0          |false          |
|WS00000004|69           |1.15               |75.0          |true           |
|WS00000005|36           |0.6                |23.4          |false          |
|WS00000006|20           |0.3333333333333333 |19.8          |false          |
|WS00000007|78           |1.3                |83.9          |true           |
|WS00000008|22           |0.36666666666666664|66.7          |false          |
|WS00000009|21           |0.35               |80.8          |true           |
|WS00000010|21           |0.35               |65.6          |fal

### 04 — Top 20 by data usage

In [8]:
exercise_04_df = sessions_df.orderBy(F.col("data_used_mb").desc_nulls_last())
exercise_04_df.show(20, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|SUB00155     |CT0069    |DV08     |2026-07-20 00:11:00|167          |97.7          |9.0              |4K            |Completed     |8881.8      |3.0        |
|WS00074513|SUB00684     |CT0056    |DV05     |2026-08-09 21:53:00|157          |96.9          |8.0              |4K            |Completed     |8408.4      |NULL       |
|WS00039201|SUB01664     |CT0027    |DV08     |2026-05-16 06:35:00|161          |97.0          |9.0              |4K            |Completed     |8282.0

## Part B — Data Quality

### 05 — Null counts in five columns

In [9]:
exercise_05_df = sessions_df.agg(
 F.sum(F.col("subscriber_id").isNull().cast("long")).alias("missing_subscriber_id"),
 F.sum(F.col("device_id").isNull().cast("long")).alias("missing_device_id"),
 F.sum(F.col("completion_pct").isNull().cast("long")).alias("missing_completion_pct"),
 F.sum(F.col("buffering_seconds").isNull().cast("long")).alias("missing_buffering_seconds"),
 F.sum(F.col("user_rating").isNull().cast("long")).alias("missing_user_rating"))
exercise_05_df.show()

+---------------------+-----------------+----------------------+-------------------------+-------------------+
|missing_subscriber_id|missing_device_id|missing_completion_pct|missing_buffering_seconds|missing_user_rating|
+---------------------+-----------------+----------------------+-------------------------+-------------------+
|                  951|             2003|                  3513|                     4424|              77605|
+---------------------+-----------------+----------------------+-------------------------+-------------------+



### 06 — Fill null buffering_seconds in a new DataFrame

In [10]:
exercise_06_df = sessions_df.na.fill({"buffering_seconds": 0})
exercise_06_df.select("session_id", "buffering_seconds").show(20, truncate=False)
print("Original nulls remain:", sessions_df.filter(F.col("buffering_seconds").isNull()).count())

+----------+-----------------+
|session_id|buffering_seconds|
+----------+-----------------+
|WS00000001|29.0             |
|WS00000002|31.0             |
|WS00000003|26.0             |
|WS00000004|2.0              |
|WS00000005|31.0             |
|WS00000006|202.0            |
|WS00000007|28.0             |
|WS00000008|11.0             |
|WS00000009|12.0             |
|WS00000010|166.0            |
|WS00000011|20.0             |
|WS00000012|22.0             |
|WS00000013|99.0             |
|WS00000014|24.0             |
|WS00000015|26.0             |
|WS00000016|4.0              |
|WS00000017|12.0             |
|WS00000018|14.0             |
|WS00000019|21.0             |
|WS00000020|0.0              |
+----------+-----------------+
only showing top 20 rows
Original nulls remain: 4424


### 07 — Exclude missing device_id and compare counts

In [11]:
exercise_07_df = sessions_df.na.drop(subset=["device_id"])
print("Before:", sessions_df.count(), "After:", exercise_07_df.count(), "Removed:", sessions_df.count() - exercise_07_df.count())

Before: 80000 After: 77997 Removed: 2003


### 08 — Completion bands using when/otherwise

In [12]:
exercise_08_df = sessions_df.withColumn("completion_band", F.when(F.col("completion_pct").isNull(), "Unknown").when(F.col("completion_pct") < 30, "Low").when(F.col("completion_pct") < 90, "Medium").otherwise("High"))
exercise_08_df.select("session_id", "completion_pct", "completion_band").show(20, truncate=False)

+----------+--------------+---------------+
|session_id|completion_pct|completion_band|
+----------+--------------+---------------+
|WS00000001|11.8          |Low            |
|WS00000002|56.4          |Medium         |
|WS00000003|63.0          |Medium         |
|WS00000004|75.0          |Medium         |
|WS00000005|23.4          |Low            |
|WS00000006|19.8          |Low            |
|WS00000007|83.9          |Medium         |
|WS00000008|66.7          |Medium         |
|WS00000009|80.8          |Medium         |
|WS00000010|65.6          |Medium         |
|WS00000011|36.1          |Medium         |
|WS00000012|78.6          |Medium         |
|WS00000013|51.6          |Medium         |
|WS00000014|28.1          |Low            |
|WS00000015|16.4          |Low            |
|WS00000016|28.0          |Low            |
|WS00000017|84.3          |Medium         |
|WS00000018|83.3          |Medium         |
|WS00000019|90.3          |High           |
|WS00000020|27.0          |Low  

## Part C — Joins and Aggregations

### 09 — Session counts and total watch minutes by genre

In [13]:
sessions_content_df = sessions_df.join(content_df, "content_id", "left")
exercise_09_df = sessions_content_df.groupBy("genre").agg(F.count("*").alias("session_count"), F.sum("watch_minutes").alias("total_watch_minutes")).orderBy(F.desc("total_watch_minutes"))
exercise_09_df.show(truncate=False)

+-----------+-------------+-------------------+
|genre      |session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|Adventure  |15561        |839700             |
|Drama      |10733        |519872             |
|Animation  |11886        |490335             |
|Comedy     |9105         |452563             |
|Sci-Fi     |9344         |433946             |
|Sports     |9967         |383044             |
|Documentary|7432         |256626             |
|Thriller   |5972         |248430             |
+-----------+-------------+-------------------+



### 10 — Average buffering and watch minutes by device_type

In [14]:
sessions_device_df = sessions_df.join(devices_df, "device_id", "left")
exercise_10_df = sessions_device_df.groupBy("device_type").agg(F.avg("buffering_seconds").alias("average_buffering_seconds"), F.avg("watch_minutes").alias("average_watch_minutes")).orderBy("device_type")
exercise_10_df.show(truncate=False)

+---------------+-------------------------+---------------------+
|device_type    |average_buffering_seconds|average_watch_minutes|
+---------------+-------------------------+---------------------+
|NULL           |35.57619248628113        |44.7866344605475     |
|Android Phone  |35.676486693680864       |45.0885616509385     |
|Desktop Browser|36.02889125799574        |45.571818733562615   |
|Game Console   |35.51243726816496        |45.04469158686026    |
|Laptop Browser |36.930938301907844       |45.84465713083719    |
|Smart TV       |36.67392739273927        |45.33776623376623    |
|Streaming Stick|36.320860309448044       |45.08570541351827    |
|Tablet         |35.23614616481949        |45.38943894389439    |
|iPhone         |35.761106907361366       |45.226752503576535   |
+---------------+-------------------------+---------------------+



### 11 — Completion rate by plan, excluding null subscriber IDs

In [15]:
sessions_subscriber_df = sessions_df.filter(F.col("subscriber_id").isNotNull()).join(subscribers_df, "subscriber_id", "left").join(plans_df, "plan_id", "left")
exercise_11_df = sessions_subscriber_df.groupBy("plan_name").agg(F.count("*").alias("all_sessions"), F.sum((F.col("session_status") == "Completed").cast("long")).alias("completed_sessions")).withColumn("completion_rate_pct", F.col("completed_sessions") / F.col("all_sessions") * 100)
exercise_11_df.show(truncate=False)

+---------+------------+------------------+-------------------+
|plan_name|all_sessions|completed_sessions|completion_rate_pct|
+---------+------------+------------------+-------------------+
|Premium  |12198       |975               |7.993113625184456  |
|NULL     |571         |42                |7.355516637478108  |
|Basic    |25086       |2006              |7.996492067288527  |
|Mobile   |17510       |1429              |8.16105082809823   |
|Standard |23684       |1868              |7.8871812193886175 |
+---------+------------+------------------+-------------------+



### 12 — Monthly recurring revenue by plan for Active subscribers

In [18]:
active_subscribers_df = subscribers_df.filter(F.col("subscriber_status") == "Active")
exercise_12_df = active_subscribers_df.join(plans_df, "plan_id", "left").groupBy("plan_name").agg(F.countDistinct("subscriber_id").alias("active_subscriber_count"), F.sum("monthly_fee_inr").alias("estimated_monthly_revenue")).orderBy(F.desc("estimated_monthly_revenue"))
exercise_12_df.show(truncate=False)

+---------+-----------------------+-------------------------+
|plan_name|active_subscriber_count|estimated_monthly_revenue|
+---------+-----------------------+-------------------------+
|Standard |604                    |240996                   |
|Premium  |326                    |211574                   |
|Basic    |632                    |157368                   |
|Mobile   |443                    |66007                    |
+---------+-----------------------+-------------------------+



### 13 — Unmatched non-null subscriber and device IDs

In [19]:
unmatched_subscribers_df = sessions_df.filter(F.col("subscriber_id").isNotNull()).join(subscribers_df.select("subscriber_id").distinct(), "subscriber_id", "left_anti")
unmatched_devices_df = sessions_df.filter(F.col("device_id").isNotNull()).join(devices_df.select("device_id").distinct(), "device_id", "left_anti")
print("Unmatched subscribers:", unmatched_subscribers_df.count())
print("Unmatched devices:", unmatched_devices_df.count())

Unmatched subscribers: 571
Unmatched devices: 481


## Part D — Date and Window Functions

### 14 — Monthly session counts and watch minutes for 2026

In [20]:
timestamp_sessions_df = sessions_df.withColumn("session_start", F.to_timestamp("session_start"))
exercise_14_df = timestamp_sessions_df.filter(F.year("session_start") == 2026).withColumn("session_month", F.date_format("session_start", "yyyy-MM")).groupBy("session_month").agg(F.count("*").alias("session_count"), F.sum("watch_minutes").alias("total_watch_minutes")).orderBy("session_month")
exercise_14_df.show(truncate=False)

+-------------+-------------+-------------------+
|session_month|session_count|total_watch_minutes|
+-------------+-------------+-------------------+
|2026-01      |9070         |407215             |
|2026-02      |8218         |372726             |
|2026-03      |9019         |403971             |
|2026-04      |8817         |399432             |
|2026-05      |9006         |407386             |
|2026-06      |8776         |404203             |
|2026-07      |9204         |416104             |
|2026-08      |8996         |407374             |
|2026-09      |8894         |406105             |
+-------------+-------------+-------------------+



### 15 — Days between signup and viewing session; earliest 10

In [21]:
signup_df = subscribers_df.select("subscriber_id", F.to_date("signup_date").alias("signup_date"))
sessions_signup_df = timestamp_sessions_df.join(signup_df, "subscriber_id", "left").withColumn("days_since_signup", F.datediff(F.to_date("session_start"), "signup_date"))
exercise_15_df = sessions_signup_df.orderBy("session_start", "session_id").select("session_id", "subscriber_id", "signup_date", "session_start", "days_since_signup")
exercise_15_df.show(10, truncate=False)

+----------+-------------+-----------+-------------------+-----------------+
|session_id|subscriber_id|signup_date|session_start      |days_since_signup|
+----------+-------------+-----------+-------------------+-----------------+
|WS00035235|SUB01707     |2024-02-13 |2026-01-01 00:00:00|688              |
|WS00032058|SUB01248     |2024-03-17 |2026-01-01 00:04:00|655              |
|WS00074739|SUB02206     |2025-08-31 |2026-01-01 00:04:00|123              |
|WS00048560|SUB02342     |2025-08-25 |2026-01-01 00:14:00|129              |
|WS00067087|SUB00563     |2024-08-11 |2026-01-01 00:15:00|508              |
|WS00004245|SUB00711     |2024-10-26 |2026-01-01 00:18:00|432              |
|WS00041710|SUB00261     |2025-05-15 |2026-01-01 00:21:00|231              |
|WS00045132|SUB02167     |2025-11-22 |2026-01-01 00:28:00|40               |
|WS00036438|SUB01237     |2024-02-01 |2026-01-01 00:41:00|700              |
|WS00030050|SUB00646     |2024-11-20 |2026-01-01 00:51:00|407              |

### 16 — Weekday/weekend average watch minutes

In [22]:
exercise_16_df = timestamp_sessions_df.withColumn("day_type", F.when(F.dayofweek("session_start").isin(1, 7), "Weekend").otherwise("Weekday")).groupBy("day_type").agg(F.avg("watch_minutes").alias("average_watch_minutes"), F.count("*").alias("session_count"))
exercise_16_df.show()

+--------+---------------------+-------------+
|day_type|average_watch_minutes|session_count|
+--------+---------------------+-------------+
| Weekday|    45.26460823373174|        57228|
| Weekend|   45.411601967328295|        22772|
+--------+---------------------+-------------+



### 17 — Top three titles by watch minutes within genre

In [23]:
genre_title_df = sessions_content_df.groupBy("genre", "content_id", "content_title").agg(F.sum("watch_minutes").alias("total_watch_minutes"))
w17 = Window.partitionBy("genre").orderBy(F.desc("total_watch_minutes"))
exercise_17_df = genre_title_df.withColumn("title_rank", F.dense_rank().over(w17)).filter(F.col("title_rank") <= 3).orderBy("genre", "title_rank")
exercise_17_df.show(truncate=False)

+-----------+----------+----------------+-------------------+----------+
|genre      |content_id|content_title   |total_watch_minutes|title_rank|
+-----------+----------+----------------+-------------------+----------+
|Adventure  |CT0105    |Infinite Horizon|66395              |1         |
|Adventure  |CT0048    |Lost Journey    |61823              |2         |
|Adventure  |CT0042    |Solar Memory    |60540              |3         |
|Animation  |CT0106    |Last River      |51398              |1         |
|Animation  |CT0114    |Emerald Circuit |51310              |2         |
|Animation  |CT0100    |Golden Puzzle   |50515              |3         |
|Comedy     |CT0027    |Blue Horizon    |61428              |1         |
|Comedy     |CT0019    |Bright Valley   |60921              |2         |
|Comedy     |CT0059    |Silver Valley   |60422              |3         |
|Documentary|CT0077    |Fading Signal   |47085              |1         |
|Documentary|CT0057    |Emerald Valley  |40819     

### 18 — Most recent session per non-null subscriber

In [24]:
w18 = Window.partitionBy("subscriber_id").orderBy(F.col("session_start").desc_nulls_last(), F.col("session_id").desc())
exercise_18_df = timestamp_sessions_df.filter(F.col("subscriber_id").isNotNull()).withColumn("row_num", F.row_number().over(w18)).filter(F.col("row_num") == 1).drop("row_num")
exercise_18_df.show(20, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00020822|SUB00001     |CT0073    |DV03     |2026-09-11 19:19:00|34           |89.5          |145.0            |HD            |Interrupted   |418.3       |NULL       |
|WS00032744|SUB00002     |CT0076    |DV05     |2026-09-02 05:41:00|45           |73.8          |219.0            |Full HD       |Interrupted   |1184.1      |NULL       |
|WS00006110|SUB00003     |CT0004    |DV03     |2026-09-30 03:13:00|11           |15.3          |3.0              |4K            |Abandoned     |591.1 

### 19 — Previous session watch minutes using lag()

In [25]:
w19 = Window.partitionBy("subscriber_id").orderBy("session_start", "session_id")
exercise_19_df = timestamp_sessions_df.filter(F.col("subscriber_id").isNotNull()).withColumn("previous_watch_minutes", F.lag("watch_minutes").over(w19)).withColumn("watch_minutes_change", F.col("watch_minutes") - F.col("previous_watch_minutes"))
exercise_19_df.select("subscriber_id", "session_start", "session_id", "watch_minutes", "previous_watch_minutes", "watch_minutes_change").show(30, truncate=False)

+-------------+-------------------+----------+-------------+----------------------+--------------------+
|subscriber_id|session_start      |session_id|watch_minutes|previous_watch_minutes|watch_minutes_change|
+-------------+-------------------+----------+-------------+----------------------+--------------------+
|SUB00001     |2026-01-02 17:32:00|WS00064390|58           |NULL                  |NULL                |
|SUB00001     |2026-01-05 04:51:00|WS00045183|78           |58                    |20                  |
|SUB00001     |2026-01-09 08:46:00|WS00028484|4            |78                    |-74                 |
|SUB00001     |2026-01-24 19:41:00|WS00043555|108          |4                     |104                 |
|SUB00001     |2026-02-18 07:43:00|WS00041862|79           |108                   |-29                 |
|SUB00001     |2026-02-25 07:38:00|WS00059616|18           |79                    |-61                 |
|SUB00001     |2026-03-13 07:34:00|WS00042345|44       

## Part E — Union, Caching and Partitioning

### 20 — Union both batches and deduplicate session_id

In [26]:
combined_sessions_df = sessions_df.unionByName(extra_df, allowMissingColumns=True)
unique_sessions_df = combined_sessions_df.dropDuplicates(["session_id"])
print("Main:", sessions_df.count(), "Extra:", extra_df.count(), "Union:", combined_sessions_df.count(), "Unique:", unique_sessions_df.count(), "Duplicates removed:", combined_sessions_df.count() - unique_sessions_df.count())

Main: 80000 Extra: 10000 Union: 90000 Unique: 89400 Duplicates removed: 600


### 21 — Lazy filter/join/withColumn/groupBy/orderBy pipeline

In [27]:
exercise_21_df = sessions_df.filter(F.col("session_status") == "Completed").join(content_df.select("content_id", "genre"), "content_id", "left").withColumn("watch_hours", F.col("watch_minutes") / 60).groupBy("genre").agg(F.sum("watch_hours").alias("total_watch_hours"), F.count("*").alias("completed_sessions")).orderBy(F.desc("total_watch_hours"))
exercise_21_df.explain()
exercise_21_df.show(10, truncate=False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_watch_hours#1251 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_watch_hours#1251 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2857]
      +- HashAggregate(keys=[genre#100], functions=[sum(watch_hours#1250), count(1)])
         +- Exchange hashpartitioning(genre#100, 200), ENSURE_REQUIREMENTS, [plan_id=2854]
            +- HashAggregate(keys=[genre#100], functions=[partial_sum(watch_hours#1250), partial_count(1)])
               +- Project [genre#100, (cast(watch_minutes#22 as double) / 60.0) AS watch_hours#1250]
                  +- BroadcastHashJoin [content_id#19], [content_id#98], LeftOuter, BuildRight, false
                     :- Project [content_id#19, watch_minutes#22]
                     :  +- Filter (isnotnull(session_status#26) AND (session_status#26 = Completed))
                     :     +- FileScan csv [content_id#19,watch_minutes#22,session_status#26] Batched: false, DataFi

### 22 — Cache completed sessions; time count and aggregation twice

In [28]:
completed_cached_df = sessions_df.filter(F.col("session_status") == "Completed").cache()
start = time.time(); count_1 = completed_cached_df.count(); count_time_1 = time.time() - start
start = time.time(); count_2 = completed_cached_df.count(); count_time_2 = time.time() - start
start = time.time(); watch_1 = completed_cached_df.agg(F.sum("watch_minutes")).first()[0]; agg_time_1 = time.time() - start
start = time.time(); watch_2 = completed_cached_df.agg(F.sum("watch_minutes")).first()[0]; agg_time_2 = time.time() - start
print("is_cached:", completed_cached_df.is_cached)
print("Counts:", count_1, count_2, "times:", round(count_time_1, 3), round(count_time_2, 3))
print("Watch totals:", watch_1, watch_2, "times:", round(agg_time_1, 3), round(agg_time_2, 3))
completed_cached_df.unpersist()

is_cached: True
Counts: 6392 6392 times: 1.294 0.143
Watch totals: 491599 491599 times: 0.327 0.146


DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

### 23 — Persist 4K sessions, perform two actions, unpersist

In [29]:
four_k_df = sessions_df.filter(F.col("stream_quality") == "4K").persist(StorageLevel.MEMORY_AND_DISK)
print("Count:", four_k_df.count())
print("Sample rows:", len(four_k_df.take(10)))
print("Storage level:", four_k_df.storageLevel)
four_k_df.unpersist()

Count: 7312
Sample rows: 10
Storage level: Disk Memory Serialized 1x Replicated


DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

### 24 — Repartition into eight and count rows per partition

In [30]:
exercise_24_df = sessions_df.repartition(8).withColumn("partition_id", F.spark_partition_id())
exercise_24_df.groupBy("partition_id").count().orderBy("partition_id").show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|10000|
|           1| 9999|
|           2|10001|
|           3|10000|
|           4|10000|
|           5|10000|
|           6|10000|
|           7|10000|
+------------+-----+



### 25 — Compare coalesce(2)/repartition(2) and write Parquet

In [31]:
sessions_8_df = sessions_df.repartition(8)
coalesced_2_df = sessions_8_df.coalesce(2)
repartitioned_2_df = sessions_8_df.repartition(2)
print("Starting:", sessions_8_df.rdd.getNumPartitions(), "Coalesce:", coalesced_2_df.rdd.getNumPartitions(), "Repartition:", repartitioned_2_df.rdd.getNumPartitions())
print("Coalesce plan:"); coalesced_2_df.explain()
print("Repartition plan:"); repartitioned_2_df.explain()
chosen_output_df = coalesced_2_df
parquet_path = "/content/streaming_sessions_parquet"
chosen_output_df.write.mode("overwrite").parquet(parquet_path)
print("Saved:", parquet_path)
print([p.name for p in Path(parquet_path).iterdir()])

Starting: 8 Coalesce: 2 Repartition: 2
Coalesce plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 2
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=3382]
            +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/watch_sessions_80000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<session_id:string,subscriber_id:string,content_id:string,device_id:string,session_start:ti...
+- == Initial Plan ==
   Coalesce 2
   +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=3382]
      +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,wat

## Final Capstone — Streaming Performance Report

In [32]:
capstone_union_df = sessions_df.unionByName(extra_df, allowMissingColumns=True)
capstone_unique_df = capstone_union_df.dropDuplicates(["session_id"])
print("Union rows:", capstone_union_df.count(), "Unique sessions:", capstone_unique_df.count())

Union rows: 90000 Unique sessions: 89400


In [33]:
capstone_dated_df = capstone_unique_df.withColumn("session_start", F.to_timestamp("session_start")).withColumn("month", F.date_format("session_start", "yyyy-MM")).filter(F.col("session_start").isNotNull())

In [34]:
capstone_enriched_df = capstone_dated_df.join(subscribers_df.select("subscriber_id", "region", "plan_id"), "subscriber_id", "left").join(content_df.select("content_id", "content_title", "genre"), "content_id", "left").join(devices_df.select("device_id", "device_type"), "device_id", "left").join(plans_df.select("plan_id", "plan_name"), "plan_id", "left")

In [35]:
capstone_report_df = capstone_enriched_df.groupBy("month", "region", "genre").agg(
    F.count("*").alias("session_count"),
    F.sum("watch_minutes").alias("total_watch_minutes"),
    (F.sum(F.when(F.col("session_status") == "Completed", 1).otherwise(0)) / F.count("*") * 100).alias("completion_rate_pct"),
    F.avg("buffering_seconds").alias("average_buffering_seconds"),
    F.sum("data_used_mb").alias("total_data_used_mb")
).withColumn("total_watch_hours", F.col("total_watch_minutes") / 60)

In [36]:
title_hours_df = capstone_enriched_df.groupBy("month", "region", "genre", "content_id", "content_title").agg((F.sum("watch_minutes") / 60).alias("title_watch_hours"))
w_titles = Window.partitionBy("month", "region", "genre").orderBy(F.desc("title_watch_hours"))
top_titles_df = title_hours_df.withColumn("title_rank", F.dense_rank().over(w_titles)).filter(F.col("title_rank") <= 3)
top_titles_grouped_df = top_titles_df.groupBy("month", "region", "genre").agg(F.collect_list(F.struct("title_rank", "content_title", "title_watch_hours")).alias("top_three_titles"))

In [37]:
capstone_final_df = capstone_report_df.join(top_titles_grouped_df, ["month", "region", "genre"], "left").cache()

In [38]:
capstone_final_df.orderBy("month", "region", "genre").show(50, truncate=False)
print("Report rows:", capstone_final_df.count())
print("Total watch hours:", capstone_final_df.agg(F.sum("total_watch_hours")).first()[0])

+-------+-------+-----------+-------------+-------------------+-------------------+-------------------------+------------------+------------------+------------------------------------------------------------------------------------------------------------------------+
|month  |region |genre      |session_count|total_watch_minutes|completion_rate_pct|average_buffering_seconds|total_data_used_mb|total_watch_hours |top_three_titles                                                                                                        |
+-------+-------+-----------+-------------+-------------------+-------------------+-------------------------+------------------+------------------+------------------------------------------------------------------------------------------------------------------------+
|2026-01|NULL   |Adventure  |37           |2301               |8.108108108108109  |37.42424242424242        |44063.59999999999 |38.35             |NULL                                          

In [39]:
capstone_path = "/content/streaming_performance_report_parquet"
capstone_final_df.repartition(3).write.mode("overwrite").parquet(capstone_path)
print("Saved report:", capstone_path)
print("Output files:", [p.name for p in Path(capstone_path).iterdir()])
capstone_final_df.unpersist()

Saved report: /content/streaming_performance_report_parquet
Output files: ['part-00001-8241517a-b967-4a78-9c6f-de5ac852f488-c000.snappy.parquet', '_SUCCESS', 'part-00000-8241517a-b967-4a78-9c6f-de5ac852f488-c000.snappy.parquet', '.part-00002-8241517a-b967-4a78-9c6f-de5ac852f488-c000.snappy.parquet.crc', 'part-00002-8241517a-b967-4a78-9c6f-de5ac852f488-c000.snappy.parquet', '.part-00001-8241517a-b967-4a78-9c6f-de5ac852f488-c000.snappy.parquet.crc', '._SUCCESS.crc', '.part-00000-8241517a-b967-4a78-9c6f-de5ac852f488-c000.snappy.parquet.crc']


DataFrame[month: string, region: string, genre: string, session_count: bigint, total_watch_minutes: bigint, completion_rate_pct: double, average_buffering_seconds: double, total_data_used_mb: double, total_watch_hours: double, top_three_titles: array<struct<title_rank:int,content_title:string,title_watch_hours:double>>]